In [1]:
import requests
import time
import csv
import json

# =========================
# CONFIG
# =========================
REQUEST_DELAY = 1.0
TARGET_DOCS = 750
POSTS_PER_CALL = 100

MIN_POST_LEN = 120
MIN_COMMENTS = 2

HEADERS = {
    "User-Agent": "academic-dev-sentiment-research/1.0"
}

# -------------------------
# TIME PERIODS (UTC)
# -------------------------
PERIODS = {
    "during_covid": (1583020800, 1625097600),
    "post_covid":   (1625097600, 1672444800),
    "ai_era":       (1672444800, 1735689600),
}

# -------------------------
# SUBREDDITS (Pushshift friendly)
# -------------------------
SUBREDDITS = [
    "developersindia",
    "cscareerquestions",
    "experienceddevs",
    "itcareerquestions",
    "programming",
    "learnprogramming",
    "careerguidance",
    "jobs",
    "recruitinghell",
    "antiwork",
    "technology",
    "machinelearning",
    "datascience"
]

BOT_PHRASES = [
    "i am a bot",
    "performed automatically",
    "contact the moderators",
    "this action was performed",
]

# =========================
# HELPERS
# =========================
def clean(text):
    return " ".join(text.replace("\n", " ").split())

def is_bot(text):
    t = text.lower()
    return any(p in t for p in BOT_PHRASES)

# =========================
# PUSHSHIFT API
# =========================
def fetch_posts(subreddit, after, before, size=100):
    url = "https://api.pushshift.io/reddit/search/submission/"
    params = {
        "subreddit": subreddit,
        "after": after,
        "before": before,
        "size": size,
        "sort": "desc",
        "sort_type": "created_utc",
    }

    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=20)
        r.raise_for_status()
        return r.json().get("data", [])
    except requests.exceptions.RequestException:
        return []

def fetch_comments(post_id):
    url = "https://api.pushshift.io/reddit/comment/search/"
    params = {
        "link_id": post_id,
        "size": 10,
        "sort": "desc",
    }

    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=20)
        r.raise_for_status()
    except requests.exceptions.RequestException:
        return []

    comments = []
    for c in r.json().get("data", []):
        body = clean(c.get("body", ""))
        if body and not is_bot(body):
            comments.append(body)
        if len(comments) == 2:
            break

    return comments

# =========================
# MAIN COLLECTION
# =========================
rows = []
doc_id = 0

print("\n🚀 Starting Pushshift Reddit collection\n")

for period, (start_ts, end_ts) in PERIODS.items():
    print(f"📅 Period: {period}")

    for subreddit in SUBREDDITS:
        if len(rows) >= TARGET_DOCS:
            break

        print(f"  📍 r/{subreddit}")
        before_cursor = end_ts

        while len(rows) < TARGET_DOCS:
            posts = fetch_posts(subreddit, start_ts, before_cursor, POSTS_PER_CALL)
            if not posts:
                break

            for post in posts:
                title = clean(post.get("title", ""))
                body = clean(post.get("selftext", ""))
                post_text = f"{title}. {body}".strip()

                if len(post_text) < MIN_POST_LEN:
                    continue
                if post.get("num_comments", 0) < MIN_COMMENTS:
                    continue

                comments = fetch_comments(post["id"])
                if not comments:
                    continue

                rows.append({
                    "doc_id": doc_id,
                    "period": period,
                    "subreddit": subreddit,
                    "post_text": post_text,
                    "comment_1": comments[0],
                    "comment_2": comments[1] if len(comments) > 1 else ""
                })

                doc_id += 1

                if len(rows) % 50 == 0:
                    print(f"    ✅ Collected {len(rows)} docs")

                if len(rows) >= TARGET_DOCS:
                    break

            before_cursor = posts[-1]["created_utc"] - 1
            time.sleep(REQUEST_DELAY)

print(f"\n🎉 DONE — Collected {len(rows)} documents")

# =========================
# SAVE FILES
# =========================
with open("reddit_temporal_pushshift.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "doc_id",
            "period",
            "subreddit",
            "post_text",
            "comment_1",
            "comment_2"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

with open("reddit_temporal_pushshift.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print("\n📁 Files saved:")
print(" - reddit_temporal_pushshift.csv")
print(" - reddit_temporal_pushshift.json")



🚀 Starting Pushshift Reddit collection

📅 Period: during_covid
  📍 r/developersindia
  📍 r/cscareerquestions
  📍 r/experienceddevs
  📍 r/itcareerquestions
  📍 r/programming
  📍 r/learnprogramming
  📍 r/careerguidance
  📍 r/jobs
  📍 r/recruitinghell
  📍 r/antiwork
  📍 r/technology
  📍 r/machinelearning
  📍 r/datascience
📅 Period: post_covid
  📍 r/developersindia
  📍 r/cscareerquestions
  📍 r/experienceddevs
  📍 r/itcareerquestions
  📍 r/programming
  📍 r/learnprogramming
  📍 r/careerguidance
  📍 r/jobs
  📍 r/recruitinghell
  📍 r/antiwork
  📍 r/technology
  📍 r/machinelearning
  📍 r/datascience
📅 Period: ai_era
  📍 r/developersindia
  📍 r/cscareerquestions
  📍 r/experienceddevs
  📍 r/itcareerquestions
  📍 r/programming
  📍 r/learnprogramming
  📍 r/careerguidance
  📍 r/jobs
  📍 r/recruitinghell
  📍 r/antiwork
  📍 r/technology
  📍 r/machinelearning
  📍 r/datascience

🎉 DONE — Collected 0 documents

📁 Files saved:
 - reddit_temporal_pushshift.csv
 - reddit_temporal_pushshift.json
